In [11]:
import requests
import pandas as pd
from datetime import datetime, timedelta
import json
import numpy as np
import joblib
import numpy as np
from sklearn.preprocessing import MinMaxScaler
from tensorflow.keras.models import load_model

In [12]:
loaded_model = load_model("../models/lstm_drought_model.keras")

In [13]:
def fetch_weather_data():
    # Define the past 10 days
    end_date = datetime.today()
    start_date = end_date - timedelta(days=10)
    print(f'Start Date: {start_date} \nEnd Date: {end_date}')

    url = f"https://api.open-meteo.com/v1/forecast?latitude=0.4667&longitude=35.9667&daily=temperature_2m_max,temperature_2m_min,temperature_2m_mean,precipitation_sum,wind_speed_10m_max,relative_humidity_2m_max,relative_humidity_2m_min,dew_point_2m_max,surface_pressure_max&timezone=Africa/Nairobi&start_date={start_date.date()}&end_date={end_date.date()}"


    response = requests.get(url)
    data = response.json()
    print(data)


    with open('drought_data.json', 'w') as f:
        json.dump(data, f)

    df = pd.DataFrame({
        "date": pd.date_range(start=start_date, periods=11),
        "temperature": data["daily"]["temperature_2m_mean"],
        "precipitation": data["daily"]["precipitation_sum"],
        "humidity": data["daily"]["relative_humidity_2m_max"],
        "wind_speed": data["daily"]["wind_speed_10m_max"],
        "dew_point": data["daily"]["dew_point_2m_max"],
        "pressure": (np.array(data["daily"]["surface_pressure_max"]) / 30),
    })
    
    return df

In [14]:
print("Hello There")
weather_data = fetch_weather_data()
print("Done")

Hello There
Start Date: 2025-02-15 14:32:33.938449 
End Date: 2025-02-25 14:32:33.938449
{'latitude': 0.5, 'longitude': 36.0, 'generationtime_ms': 0.11968612670898438, 'utc_offset_seconds': 10800, 'timezone': 'Africa/Nairobi', 'timezone_abbreviation': 'GMT+3', 'elevation': 1101.0, 'daily_units': {'time': 'iso8601', 'temperature_2m_max': '°C', 'temperature_2m_min': '°C', 'temperature_2m_mean': '°C', 'precipitation_sum': 'mm', 'wind_speed_10m_max': 'km/h', 'relative_humidity_2m_max': '%', 'relative_humidity_2m_min': '%', 'dew_point_2m_max': '°C', 'surface_pressure_max': 'hPa'}, 'daily': {'time': ['2025-02-15', '2025-02-16', '2025-02-17', '2025-02-18', '2025-02-19', '2025-02-20', '2025-02-21', '2025-02-22', '2025-02-23', '2025-02-24', '2025-02-25'], 'temperature_2m_max': [36.0, 36.1, 35.5, 34.9, 35.1, 36.0, 35.4, 35.7, 35.4, 35.4, 35.1], 'temperature_2m_min': [20.1, 18.6, 18.0, 16.0, 16.7, 15.5, 16.3, 16.8, 16.9, 17.7, 17.4], 'temperature_2m_mean': [28.3, 28.0, 27.5, 26.1, 26.1, 26.2, 26.

In [15]:
# features = ['temperature', 'humidity', 'dew_point', 'wind_speed', 'pressure', 'precipitation']
features = ['temperature', 'dew_point', 'humidity', 'wind_speed', 'pressure', 'precipitation']

# Cluster the data
kmeans = joblib.load("../models/kmeans_model.pkl")
weather_data["region_id"] = kmeans.predict(weather_data[features])


/home/ronny/.local/lib/python3.10/site-packages/sklearn/utils/validation.py:2732: UserWarning: X has feature names, but KMeans was fitted without feature names
  warnings.warn(


In [16]:
weather_data

,date,temperature,precipitation,humidity,wind_speed,dew_point,pressure,region_id
0,2025-02-15 14:32:33.938449,28.3,0.0,62,21.2,12.9,29.886667,3
1,2025-02-16 14:32:33.938449,28.0,0.0,72,22.8,14.4,29.840000,3
2,2025-02-17 14:32:33.938449,27.5,0.0,74,24.2,14.8,29.846667,3
3,2025-02-18 14:32:33.938449,26.1,0.0,70,18.0,10.6,29.886667,3
4,2025-02-19 14:32:33.938449,26.1,0.0,61,13.3,10.3,29.900000,3
5,2025-02-20 14:32:33.938449,26.2,0.0,59,18.8,8.4,29.863333,3
6,2025-02-21 14:32:33.938449,26.7,0.0,56,23.8,10.1,29.880000,3
7,2025-02-22 14:32:33.938449,26.7,0.0,54,25.4,9.8,29.873333,3
8,2025-02-23 14:32:33.938449,26.6,0.0,53,15.8,12.3,29.913333,3
9,2025-02-24 14:32:33.938449,27.4,0.0,70,19.1,12.8,29.926667,3


In [17]:
# Preprocess the data and scale it
scaler = joblib.load('../drought_scaler.pkl')
weather_data_scaled = scaler.transform(weather_data[features])

In [18]:
sequence_length = 10  # The LSTM model was trained with 10-day sequences
num_future_days = 5   # We want predictions for the next 7 days

X_inputs = []

# Ensure sequence length is respected
for i in range(num_future_days):
    seq = weather_data_scaled[i:i+sequence_length]  # Slicing the NumPy array
    
    if seq.shape[0] == sequence_length:  # Only include full sequences
        X_inputs.append(seq)

print(f"X Inputs: {X_inputs}")
# Convert to a NumPy array properly
X_inputs = np.array(X_inputs)

X Inputs: [array([[1.33076923, 0.78421053, 0.48571429, 0.74814815, 0.29885057,
        0.        ],
       [1.30769231, 0.86315789, 0.62857143, 0.80740741, 0.13793103,
        0.        ],
       [1.26923077, 0.88421053, 0.65714286, 0.85925926, 0.16091954,
        0.        ],
       [1.16153846, 0.66315789, 0.6       , 0.62962963, 0.29885057,
        0.        ],
       [1.16153846, 0.64736842, 0.47142857, 0.45555556, 0.34482759,
        0.        ],
       [1.16923077, 0.54736842, 0.44285714, 0.65925926, 0.2183908 ,
        0.        ],
       [1.20769231, 0.63684211, 0.4       , 0.84444444, 0.27586207,
        0.        ],
       [1.20769231, 0.62105263, 0.37142857, 0.9037037 , 0.25287356,
        0.        ],
       [1.2       , 0.75263158, 0.35714286, 0.54814815, 0.3908046 ,
        0.        ],
       [1.26153846, 0.77894737, 0.6       , 0.67037037, 0.43678161,
        0.        ]]), array([[1.30769231, 0.86315789, 0.62857143, 0.80740741, 0.13793103,
        0.        ],
       [

In [19]:
# Make predictions for each sequence
predicted_probabilities = loaded_model.predict(X_inputs)

# Convert probabilities to percentage format
drought_probabilities = [round(float(prob[0]) * 100, 2) for prob in predicted_probabilities]

1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step


In [20]:
# Display results
for i, prob in enumerate(drought_probabilities):
    print(f"Drought Probability for Day {i+1}: {prob}%")


Drought Probability for Day 1: 90.11%
Drought Probability for Day 2: 90.43%
